# Hafta 10 - Yapay Sinir Ağları (ANN) Temelleri ve MNIST

Bu defterde yapay sinir ağlarının temellerini öğrenecek ve MNIST el yazısı rakam tanıma problemi üzerinde uygulama yapacağız.

## Biyolojik Nörondan Yapay Nörona

### Biyolojik Nöron
İnsan beyni yaklaşık **86 milyar** nörondan oluşur. Her biyolojik nöron şu bileşenlere sahiptir:
- **Dendritler:** Diğer nöronlardan gelen sinyalleri alır
- **Hücre Gövdesi (Soma):** Gelen sinyalleri toplar ve işler
- **Akson:** İşlenmiş sinyali diğer nöronlara iletir
- **Sinaps:** Nöronlar arası bağlantı noktaları

### Yapay Nöron (Perceptron)
Yapay nöron, biyolojik nöronun matematiksel bir modelidir:
- **Girdiler (x₁, x₂, ..., xₙ):** Dendritler gibi veri alır
- **Ağırlıklar (w₁, w₂, ..., wₙ):** Sinaptik bağlantı güçleri
- **Toplama Fonksiyonu:** z = Σ(wᵢ · xᵢ) + b (bias)
- **Aktivasyon Fonksiyonu:** f(z) → çıktı üretir (ReLU, Sigmoid, Softmax vb.)

### Çok Katmanlı Yapay Sinir Ağı
- **Girdi Katmanı:** Veriyi alır
- **Gizli Katmanlar:** Özellik çıkarımı yapar
- **Çıktı Katmanı:** Sonucu üretir

**İleri Yayılım (Forward Propagation):** Girdiden çıktıya doğru hesaplama yapılır.

**Geri Yayılım (Backpropagation):** Hata, çıktıdan girdiye doğru yayılarak ağırlıklar güncellenir.

## 1. Kütüphanelerin Yüklenmesi

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |
| `tensorflow` | Derin öğrenme modelleri oluşturma ve eğitme |
| `warnings` | Uyarı mesajlarını yönetme |


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow sürümü: {tf.__version__}")
print(f"Keras sürümü: {keras.__version__}")

## 2. MNIST Veri Setinin Yüklenmesi

MNIST, 0-9 arası el yazısı rakamlardan oluşan klasik bir veri setidir:
- **60.000** eğitim görüntüsü
- **10.000** test görüntüsü
- Her görüntü **28x28** piksel, gri tonlamalı

In [ ]:
# MNIST veri setini yükle
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print(f"Eğitim seti boyutu: {x_train.shape}")
print(f"Eğitim etiketleri boyutu: {y_train.shape}")
print(f"Test seti boyutu: {x_test.shape}")
print(f"Test etiketleri boyutu: {y_test.shape}")
print(f"\nPiksel değer aralığı: [{x_train.min()}, {x_train.max()}]")
print(f"Etiket değerleri: {np.unique(y_train)}")

## 3. Normalizasyon

Piksel değerlerini [0, 255] aralığından [0, 1] aralığına ölçeklendiriyoruz. Bu işlem:
- Eğitimi hızlandırır
- Sayısal kararlılığı artırır
- Gradyanların daha iyi akmasını sağlar

In [ ]:
# Normalizasyon: [0, 255] -> [0, 1]
x_train = x_train / 255.0
x_test = x_test / 255.0

print(f"Normalizasyon sonrası piksel aralığı: [{x_train.min()}, {x_train.max()}]")

## 4. Örnek Rakamların Görselleştirilmesi

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Her rakamdan birer örnek göster
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('MNIST Veri Setinden Örnek Rakamlar', fontsize=14, fontweight='bold')

for digit in range(10):
    ax = axes[digit // 5, digit % 5]
    idx = np.where(y_train == digit)[0][0]
    ax.imshow(x_train[idx], cmap='gray')
    ax.set_title(f'Rakam: {digit}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Rastgele 16 örnek göster
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
fig.suptitle('Rastgele 16 Eğitim Örneği', fontsize=14, fontweight='bold')

indices = np.random.choice(len(x_train), 16, replace=False)
for i, idx in enumerate(indices):
    ax = axes[i // 4, i % 4]
    ax.imshow(x_train[idx], cmap='gray')
    ax.set_title(f'Etiket: {y_train[idx]}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. ANN Modelinin Oluşturulması

Sequential (Sıralı) model kullanarak basit bir yapay sinir ağı oluşturuyoruz:

| Katman | Açıklama |
|--------|----------|
| **Flatten** | 28x28 görüntüyü 784 boyutlu vektöre düzleştirir |
| **Dense(128, relu)** | 128 nöronlu gizli katman, ReLU aktivasyonu |
| **Dropout(0.2)** | %20 oranında rastgele nöron devre dışı bırakır (aşırı öğrenmeyi önler) |
| **Dense(10, softmax)** | 10 sınıf için olasılık çıktısı |

In [ ]:
# Modeli oluştur
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# Model özetini göster
model.summary()

## 6. Modelin Derlenmesi

- **Optimizasyon:** Adam (Adaptive Moment Estimation) - öğrenme hızını otomatik ayarlar
- **Kayıp Fonksiyonu:** Sparse Categorical Crossentropy - çok sınıflı sınıflandırma için
- **Metrik:** Accuracy (doğruluk)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model başarıyla derlendi!")

## 7. Modelin Eğitilmesi

- **Epoch:** Tüm veri seti üzerinden 10 tam geçiş
- **Validation Split:** Eğitim verisinin %20'si doğrulama için ayrılır

In [ ]:
# Modeli eğit
history = model.fit(
    x_train, y_train,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

print("\nEğitim tamamlandı!")

## 8. Eğitim Eğrilerinin Görselleştirilmesi

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Doğruluk eğrisi
ax1.plot(history.history['accuracy'], label='Eğitim Doğruluğu', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Doğrulama Doğruluğu', linewidth=2)
ax1.set_title('Model Doğruluğu', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Doğruluk')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Kayıp eğrisi
ax2.plot(history.history['loss'], label='Eğitim Kaybı', linewidth=2)
ax2.plot(history.history['val_loss'], label='Doğrulama Kaybı', linewidth=2)
ax2.set_title('Model Kaybı', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Kayıp')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Test Seti Üzerinde Değerlendirme

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test Kaybı: {test_loss:.4f}")
print(f"Test Doğruluğu: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## 10. Tahminler ve Örnek Gösterim

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Tahminleri al
predictions = model.predict(x_test)

# Rastgele 12 test örneği göster
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
fig.suptitle('Test Seti Tahminleri', fontsize=14, fontweight='bold')

indices = np.random.choice(len(x_test), 12, replace=False)
for i, idx in enumerate(indices):
    ax = axes[i // 4, i % 4]
    ax.imshow(x_test[idx], cmap='gray')
    predicted = np.argmax(predictions[idx])
    actual = y_test[idx]
    confidence = predictions[idx][predicted] * 100
    
    color = 'green' if predicted == actual else 'red'
    ax.set_title(f'Tahmin: {predicted} (Gerçek: {actual})\nGüven: %{confidence:.1f}',
                 fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 11. Karmaşıklık Matrisi (Confusion Matrix)

### Karışıklık Matrisi Görselleştirmesi

Modelin doğru ve yanlış tahminlerini ısı haritası olarak görselleştiriyoruz. Köşegen üzerindeki değerler doğru tahminleri, dışındakiler hataları gösterir.

In [ ]:
# Tahmin edilen sınıfları al
y_pred = np.argmax(predictions, axis=1)

# Karmaşıklık matrisini oluştur
cm = confusion_matrix(y_test, y_pred)

# Görselleştir
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Rakam Sınıflandırma - Karmaşıklık Matrisi', fontsize=14, fontweight='bold')
ax.set_xlabel('Tahmin Edilen Rakam', fontsize=12)
ax.set_ylabel('Gerçek Rakam', fontsize=12)
plt.tight_layout()
plt.show()

# Her rakam için doğruluk
print("\nHer rakam için doğruluk oranı:")
for digit in range(10):
    mask = y_test == digit
    acc = (y_pred[mask] == digit).mean() * 100
    print(f"  Rakam {digit}: %{acc:.1f}")

## Özet

Bu defterde şunları öğrendik:
- Biyolojik nörondan yapay nörona geçiş
- MNIST veri setini yükleme ve ön işleme
- TensorFlow/Keras ile basit bir ANN modeli oluşturma
- Modeli eğitme ve doğrulama eğrilerini analiz etme
- Test seti üzerinde değerlendirme ve karmaşıklık matrisi

**Sonraki adım:** Daha karmaşık veri setleriyle çalışma ve model mimarisini geliştirme!